In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType, TimestampType, BooleanType
import pyspark.sql.functions as f


In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
storage_account_name = dbutils.widgets.get('storage_account_name')
container_name = dbutils.widgets.get('container_name')
catalog_name = dbutils.widgets.get('catalog_name')

print(storage_account_name, container_name, catalog_name)

In [0]:
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_shipments/landing/"
bronze_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_order_shipments/"

print(f"""
      adls_path = {adls_path}
      bronze_checkpoint_path = {bronze_checkpoint_path}""")

In [0]:
spark.readStream \
    .format('cloudFiles') \
    .option('cloudFiles.format', 'csv') \
    .option('cloudFiles.schemaLocation', bronze_checkpoint_path) \
    .option('cloudFiles.schemaEvolutionMode', 'rescue') \
    .option('header', 'true') \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option('rescuedDataColumn', '_rescued_data') \
    .option("cloudFiles.includeExistingFiles", "true") \
    .option("pathGlobFilter", "*.csv") \
    .load(adls_path) \
    .withColumn("ingest_timestamp", f.current_timestamp()) \
    .withColumn('source_file', f.col('_metadata.file_path')) \
    .writeStream \
    .outputMode("append") \
    .option("checkpointLocation", bronze_checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable(f"{catalog_name}.bronze.brz_order_shipments") \
    .awaitTermination()